# Prompt Chaining Workflow

Sequential Workflow :


.When series of llm responses needed for completion of task then prompt chaining is used
.When multiple time interaction is needed with llm

In [1]:
from langgraph.graph import StateGraph , START , END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

In [4]:
# Create a state :

class BlogState(TypedDict):

    title : str
    outline : str
    content : str


In [6]:
# Tasks / Functions :

# Task 1 : 
def create_outline(state : BlogState) -> BlogState :
    # Fetch title :
    title = state['title']

    # call llm and generate ouline : 
    prompt = f'Generate a outline for a Blog on the topic - {title}'
    outline = model.invoke(prompt).content

    # update state : 
    state['outline'] = outline


    return state


In [7]:
# Task 2 : 
def create_blog(state : BlogState) -> BlogState :

    # fetch title and outline :
    title = state['title']
    outline = state['outline']

    # call the llm and generate content :
    prompt = f'write a detailed blog on title -{title} using the following outline \n {outline}'
    content = model.invoke(prompt).content

    # update state : 
    state['content'] = content

    return state

In [ ]:
# Create a  Graph  :

graph = StateGraph(BlogState)

# Add nodes :
graph.add_node('create_outline' , create_outline)
graph.add_node('create_blog' , create_blog)

# Add Edges : 
graph.add_edge(START , 'create_outline')
graph.add_edge('create_outline' , 'create_blog')
graph.add_edge('create_blog' , END)

# Compile Graph : 
workflow = graph.compile()

In [9]:
# Execute : 

initial_state = {'title' : "Rise of Messi"}
final_state = workflow.invoke(initial_state)

print(final_state)

{'title': 'Rise of Messi', 'outline': 'Here\'s a comprehensive outline for a blog post on the "Rise of Messi," designed to be engaging, informative, and celebratory.\n\n---\n\n## Blog Post Outline: The Unstoppable Ascent: Tracing the Rise of Lionel Messi\n\n**Blog Title Options:**\n*   The Unstoppable Ascent: Tracing the Rise of Lionel Messi\n*   From Rosario to Royalty: The Epic Journey of Lionel Messi\n*   Beyond the Ball: How Lionel Messi Became a Legend\n*   The Making of a Maestro: Understanding Messi\'s Rise to Greatness\n\n---\n\n**I. Introduction (Approx. 150-200 words)**\n\n*   **A. Hook:** Start with Messi\'s current status – World Cup winner, undisputed GOAT contender, global icon. Contrast this with his humble beginnings.\n    *   *Example:* "In the annals of football history, few names shine as brightly as Lionel Messi. A World Cup winner, a record-breaking Ballon d\'Or holder, and a magician with the ball, his legend is etched in gold. But before the global adoration and 

In [10]:
print(final_state['outline'])

Here's a comprehensive outline for a blog post on the "Rise of Messi," designed to be engaging, informative, and celebratory.

---

## Blog Post Outline: The Unstoppable Ascent: Tracing the Rise of Lionel Messi

**Blog Title Options:**
*   The Unstoppable Ascent: Tracing the Rise of Lionel Messi
*   From Rosario to Royalty: The Epic Journey of Lionel Messi
*   Beyond the Ball: How Lionel Messi Became a Legend
*   The Making of a Maestro: Understanding Messi's Rise to Greatness

---

**I. Introduction (Approx. 150-200 words)**

*   **A. Hook:** Start with Messi's current status – World Cup winner, undisputed GOAT contender, global icon. Contrast this with his humble beginnings.
    *   *Example:* "In the annals of football history, few names shine as brightly as Lionel Messi. A World Cup winner, a record-breaking Ballon d'Or holder, and a magician with the ball, his legend is etched in gold. But before the global adoration and the endless trophies, there was a quiet boy from Rosario, Ar

In [11]:
print(final_state['content'])


## From Rosario to Royalty: The Epic Journey of Lionel Messi

**(Image: Young Messi in Argentina kit, perhaps with a slightly blurred background of Rosario)**

In the annals of football history, few names shine as brightly as Lionel Messi. A World Cup winner, a record-breaking Ballon d'Or holder, and a magician with the ball, his legend is etched in gold. His current status as an undisputed GOAT contender and global icon feels almost preordained. But before the global adoration, the endless trophies, and the iconic moments, there was a quiet, unassuming boy from Rosario, Argentina, facing challenges that could have derailed any ordinary dream.

This blog post will explore the extraordinary journey of **Lionel Messi**, tracing his rise from his early struggles and unique talent to his unparalleled dominance and ultimate triumph. We'll highlight the key moments, the unwavering determination, and the sheer brilliance that forged a legend, offering readers a comprehensive look at how a shy